# Курсовая работа — Полный пайплайн анализа видеоконтента

## Сценарий
Автоматизированный анализ видеофайла для выявления деструктивного / нежелательного контента:
1. Нарезка видео на кадры
2. OCR субтитров → дедупликация
3. Распознавание объектов YOLO → склейка событий
4. Классификация сцен ResNet
5. Описание через LLM (Gemini)
6. Whisper-транскрипция аудио
7. Формирование итогового отчёта JSON

## Прикладная задача
Выявление деструктивного контента (насилие, оружие, нежелательная символика) в видеопотоке.

In [ ]:
!pip install opencv-python-headless easyocr ultralytics openai-whisper moviepy rapidfuzz google-generativeai torch torchvision -q
!apt-get install -y ffmpeg -q

In [ ]:
import os, json, cv2, whisper, base64, torch
import numpy as np
import pandas as pd
import easyocr
from PIL import Image
from ultralytics import YOLO
from moviepy.editor import VideoFileClip
from rapidfuzz import fuzz, process
from torchvision import models, transforms
from tqdm.notebook import tqdm

# === Пути ===
BASE_DIR    = '/content'
FRAMES_DIR  = f'{BASE_DIR}/frames'
RESULTS_DIR = f'{BASE_DIR}/results'
for d in [FRAMES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# === Параметры ===
FRAME_STEP       = 30     # каждый 30-й кадр
CONF_YOLO        = 0.4
OCR_DEDUP_THR    = 85
DET_MERGE_WINDOW = 5
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Устройство: {DEVICE}')

In [ ]:
# === Шаг 0: Загрузка видео ===
from google.colab import files
uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print(f'Загружен файл: {VIDEO_PATH}')

In [ ]:
# === Шаг 1: Нарезка на кадры (ПЗ 2) ===
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_idx, saved = 0, 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % FRAME_STEP == 0:
        cv2.imwrite(f'{FRAMES_DIR}/frame_{frame_idx:06d}.jpg', frame)
        saved += 1
    frame_idx += 1
cap.release()
FRAME_LIST = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'Кадров сохранено: {saved} | FPS: {fps:.1f} | Длительность: {total_frames/fps:.1f} сек')

In [ ]:
# === Шаг 2: OCR субтитров (ПЗ 3) ===
ocr_reader = easyocr.Reader(['ru', 'en'], gpu=(DEVICE=='cuda'))
ocr_rows = []
for fname in tqdm(FRAME_LIST, desc='OCR'):
    img = cv2.imread(f'{FRAMES_DIR}/{fname}')
    h = img.shape[0]
    crop = img[int(h*0.85):, :]
    for _, text, prob in ocr_reader.readtext(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)):
        if text.strip() and prob > 0.4:
            ocr_rows.append({'frame': fname, 'text': text.strip(), 'conf': round(prob,3)})

df_ocr = pd.DataFrame(ocr_rows)

# Дедупликация (ПЗ 8)
def dedup(texts, thr=OCR_DEDUP_THR):
    uniq = []
    for t in texts:
        if not uniq or (process.extractOne(t, uniq, scorer=fuzz.ratio) or (None,0))[1] < thr:
            uniq.append(t)
    return uniq

uniq_subtitles = dedup(df_ocr['text'].tolist())
print(f'OCR строк: {len(df_ocr)} → уникальных: {len(uniq_subtitles)}')

In [ ]:
# === Шаг 3: YOLO детекция (ПЗ 5) + склейка (ПЗ 8) ===
yolo = YOLO('yolov8n.pt')
yolo_rows = []
for fname in tqdm(FRAME_LIST, desc='YOLO'):
    for r in yolo(f'{FRAMES_DIR}/{fname}', conf=CONF_YOLO, verbose=False):
        for box in r.boxes:
            yolo_rows.append({'frame': fname,
                              'class': yolo.names[int(box.cls[0])],
                              'conf':  round(float(box.conf[0]),3),
                              'frame_num': int(fname.split('_')[1].split('.')[0])})

df_yolo = pd.DataFrame(yolo_rows)

def merge_det(g):
    g = g.sort_values('frame_num')
    evts, start, prev = [], g.iloc[0], g.iloc[0]['frame_num']
    for _, r in g.iloc[1:].iterrows():
        if r['frame_num'] - prev > DET_MERGE_WINDOW:
            evts.append({'class': start['class'], 'start_frame': start['frame_num'], 'end_frame': prev})
            start = r
        prev = r['frame_num']
    evts.append({'class': start['class'], 'start_frame': start['frame_num'], 'end_frame': prev})
    return pd.DataFrame(evts)

df_yolo_merged = df_yolo.groupby('class', group_keys=False).apply(merge_det).reset_index(drop=True)
print(f'YOLO детекций: {len(df_yolo)} → событий: {len(df_yolo_merged)}')

In [ ]:
# === Шаг 4: Whisper аудио (ПЗ 4) ===
AUDIO_PATH = f'{BASE_DIR}/audio.wav'
VideoFileClip(VIDEO_PATH).audio.write_audiofile(AUDIO_PATH, verbose=False, logger=None)
whisper_model = whisper.load_model('base')
transcript = whisper_model.transcribe(AUDIO_PATH, language='ru', verbose=False)
whisper_segments = [{'start': round(s['start'],2), 'end': round(s['end'],2), 'text': s['text'].strip()}
                    for s in transcript['segments']]
print(f'Whisper сегментов: {len(whisper_segments)}')

In [ ]:
# === Шаг 5: Итоговый отчёт ===
report = {
    'video': VIDEO_PATH,
    'duration_sec': round(total_frames / fps, 1),
    'frames_analyzed': len(FRAME_LIST),
    'ocr': {
        'total_recognized': len(df_ocr),
        'unique_after_dedup': len(uniq_subtitles),
        'sample': uniq_subtitles[:10],
    },
    'yolo': {
        'total_detections': len(df_yolo),
        'merged_events': len(df_yolo_merged),
        'top_classes': df_yolo['class'].value_counts().head(5).to_dict(),
        'events': df_yolo_merged.to_dict(orient='records'),
    },
    'whisper': {
        'total_segments': len(whisper_segments),
        'full_text_preview': transcript['text'][:500],
        'segments': whisper_segments[:10],
    },
}

REPORT_PATH = f'{RESULTS_DIR}/final_report.json'
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(f'Отчёт сохранён: {REPORT_PATH}')
print(json.dumps(report, ensure_ascii=False, indent=2)[:1000])

In [ ]:
# Скачать отчёт
from google.colab import files
files.download(REPORT_PATH)